# CNN — Vision Artificial para Clasificacion de Objetos (CIFAR-10)

| Campo | Valor |
|:------|:------|
| **Autor** | Borja Mora Mendez |
| **Contacto** | borja.mora.mendez@gmail.com |
| **LinkedIn** | [linkedin.com/in/borjamoramendez](https://linkedin.com/in/borjamoramendez) |
| **Categoria** | Machine Learning > Redes Neuronales > CNN (Vision) |
| **Dataset** | CIFAR-10 (60.000 imagenes 32x32 color, 10 clases) |
| **Ultima actualizacion** | Julio 2026 |

## 1. Contexto de negocio

Una empresa de seguridad vial desarrolla un sistema de deteccion automatica
de objetos en intersecciones urbanas. Las camaras capturan imagenes que deben
clasificarse en tiempo real para:

- **Deteccion de vehiculos:** contar trafico y optimizar semaforos.
- **Seguridad peatonal:** alertas cuando peatones cruzan fuera de paso.
- **Incidencias:** detectar vehiculos detenidos o animales en la via.

Pregunta analitica: **puede una CNN clasificar objetos en imagenes a color
con precision suficiente para un MVP de deteccion automatica?**

## 2. Objetivo y justificacion del modelo

Comparamos 3 arquitecturas CNN de complejidad creciente sobre CIFAR-10
(imagenes a color 32x32, 10 clases). A diferencia de Fashion-MNIST (grayscale
28x28), CIFAR-10 requiere procesamiento de 3 canales RGB.

| Arquitectura | Parametros | Ventaja |
|:-------------|:-----------|:--------|
| CNN Simple (2 conv) | ~60K | Rapida, baseline |
| CNN Media (3 conv + Dropout) | ~150K | Regularizada |
| CNN con BatchNorm | ~180K | Estabilidad en entrenamiento |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print(f'TensorFlow: {tf.__version__}')
print('Entorno configurado correctamente.')

## 4. Carga y exploracion inicial

### Diccionario de clases

| ID | Clase | Categoria |
|:---|:------|:----------|
| 0 | airplane | Vehiculo aereo |
| 1 | automobile | Vehiculo terrestre |
| 2 | bird | Animal |
| 3 | cat | Animal |
| 4 | deer | Animal |
| 5 | dog | Animal |
| 6 | frog | Animal |
| 7 | horse | Animal |
| 8 | ship | Vehiculo maritimo |
| 9 | truck | Vehiculo terrestre |

In [ ]:
(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.flatten()
y_test = y_test.flatten()

CLASES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
          'dog', 'frog', 'horse', 'ship', 'truck']

print(f'Train: {X_train_raw.shape} | Test: {X_test_raw.shape}')
print(f'Rango pixeles: [{X_train_raw.min()}, {X_train_raw.max()}]')
print(f'Canales: {X_train_raw.shape[-1]} (RGB)')
print(f'\nDistribucion del target (train):')
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {CLASES[u]:12s}: {c:5d} ({c/len(y_train):.1%})')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
np.random.seed(42)
for ax, idx in zip(axes.flat, np.random.choice(len(X_train_raw), 10, replace=False)):
    ax.imshow(X_train_raw[idx])
    ax.set_title(CLASES[y_train[idx]], fontsize=10)
    ax.axis('off')
plt.suptitle('Muestra del Dataset CIFAR-10 (32x32 RGB)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 5. Preparacion de datos

In [ ]:
X_train = X_train_raw.astype('float32') / 255.0
X_test  = X_test_raw.astype('float32') / 255.0

print(f'Rango normalizado: [{X_train.min():.1f}, {X_train.max():.1f}]')
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 6. Modelo principal: CNN Simple (2 capas Conv2D)

Arquitectura baseline: 2 bloques Conv2D + MaxPooling, seguidos de 1 capa densa.

In [ ]:
tf.random.set_seed(42)

cnn_simple = keras.Sequential([
    keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

cnn_simple.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn_simple.summary()

In [ ]:
h1 = cnn_simple.fit(X_train, y_train, epochs=5, batch_size=128,
                    validation_split=0.1, verbose=1)
_, acc_simple = cnn_simple.evaluate(X_test, y_test, verbose=0)
print(f'\nAccuracy CNN Simple: {acc_simple:.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for metric, ax, title in [('loss', axes[0], 'Loss'), ('accuracy', axes[1], 'Accuracy')]:
    ax.plot(h1.history[metric], label='Train')
    ax.plot(h1.history[f'val_{metric}'], label='Val')
    ax.set_title(f'{title} - CNN Simple', fontsize=12)
    ax.set_xlabel('Epoch')
    ax.legend()
plt.tight_layout()
plt.show()

## 7. CNN Media (3 Conv2D + Dropout)

Anadimos una tercera capa convolucional y Dropout (0.3) para regularizar.

In [ ]:
tf.random.set_seed(42)

cnn_media = keras.Sequential([
    keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Dropout(0.25),
    keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Dropout(0.25),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation='softmax')
])

cnn_media.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h2 = cnn_media.fit(X_train, y_train, epochs=5, batch_size=128,
                   validation_split=0.1, verbose=1)
_, acc_media = cnn_media.evaluate(X_test, y_test, verbose=0)
print(f'\nAccuracy CNN Media: {acc_media:.1%}')

## 8. CNN con BatchNormalization

BatchNorm estabiliza el entrenamiento normalizando las activaciones entre capas.

In [ ]:
tf.random.set_seed(42)

cnn_bn = keras.Sequential([
    keras.layers.Conv2D(32, (3, 3), padding='same', input_shape=(32, 32, 3)),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.Conv2D(32, (3, 3), padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation='softmax')
])

cnn_bn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h3 = cnn_bn.fit(X_train, y_train, epochs=5, batch_size=128,
                validation_split=0.1, verbose=1)
_, acc_bn = cnn_bn.evaluate(X_test, y_test, verbose=0)
print(f'\nAccuracy CNN BatchNorm: {acc_bn:.1%}')

## 9. Comparativa visual

In [ ]:
# -- Confusion del mejor modelo --
best_model = cnn_bn if acc_bn >= acc_media else cnn_media
best_name = 'CNN BatchNorm' if acc_bn >= acc_media else 'CNN Media'
y_pred = np.argmax(best_model.predict(X_test, verbose=0), axis=1)

fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=CLASES, cmap='Blues', ax=ax,
    xticks_rotation=45
)
ax.set_title(f'Matriz de Confusion - {best_name}', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# -- Learning curves comparativas --
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for h, name, color in [(h1, 'Simple', 'steelblue'), (h2, 'Media+Drop', 'darkorange'), (h3, 'BatchNorm', 'forestgreen')]:
    axes[0].plot(h.history['val_loss'], label=name, color=color, linewidth=2)
    axes[1].plot(h.history['val_accuracy'], label=name, color=color, linewidth=2)
axes[0].set_title('Validation Loss por Arquitectura', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[1].set_title('Validation Accuracy por Arquitectura', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# -- Barras comparativas --
models = ['CNN Simple', 'CNN Media+Drop', 'CNN BatchNorm']
accs = [acc_simple, acc_media, acc_bn]
colors = ['steelblue', 'darkorange', 'forestgreen']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(models, accs, color=colors, alpha=0.85)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.1%}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0.4, 0.9)
ax.set_title('Accuracy Comparativa - Arquitecturas CNN (CIFAR-10)', fontsize=13)
ax.set_ylabel('Accuracy')
plt.tight_layout()
plt.show()

In [ ]:
# -- Ejemplos dificiles --
errores = np.where(y_pred != y_test)[0]
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
np.random.seed(42)
for ax, e in zip(axes.flat, np.random.choice(errores, 10, replace=False)):
    ax.imshow(X_test_raw[e])
    ax.set_title(f'Real: {CLASES[y_test[e]]}\nPred: {CLASES[y_pred[e]]}',
                 fontsize=8, color='red')
    ax.axis('off')
plt.suptitle('Clasificaciones Incorrectas', fontsize=13, y=1.05)
plt.tight_layout()
plt.show()